In [ ]:
# url = 'https://raw.githubusercontent.com/luishrufino/obesity-predict-model/main/Obesity.csv'
# obesity_df = pd.read_csv(url)

pd.set_option('display.max_columns', None)

obesity_df = pd.read_csv('Obesity.csv', sep=',')

obesity_df.describe().transpose()

## Dicionário de Dados – Base `obesity.csv`

### 🔢 Colunas Numéricas

<table>
  <thead>
    <tr>
      <th style="width:150px">Coluna</th>
      <th style="width:500px">Descrição</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><b>AGE</b></td><td>Idade da pessoa (anos).</td></tr>
    <tr><td><b>HEIGHT</b></td><td>Altura em metros.</td></tr>
    <tr><td><b>WEIGHT</b></td><td>Peso em quilogramas (kg).</td></tr>
    <tr><td><b>FCVC</b></td><td>Frequência no consumo de vegetais (escala de 1 a 3).</td></tr>
    <tr><td><b>NCP</b></td><td>Número de refeições principais por dia.</td></tr>
    <tr><td><b>CH20</b></td><td>Consumo diário de água (litros).</td></tr>
    <tr><td><b>FAF</b></td><td>Frequência de atividade física semanal (horas).</td></tr>
    <tr><td><b>TUE</b></td><td>Tempo de uso de dispositivos eletrônicos por dia (horas).</td></tr>
  </tbody>
</table>

---

### ✅ Colunas Binárias (Yes/No)

<table>
  <thead>
    <tr>
      <th style="width:150px">Coluna</th>
      <th style="width:500px">Descrição</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><b>FAMILY_HISTORY</b></td><td>Histórico familiar de sobrepeso.</td></tr>
    <tr><td><b>FAVC</b></td><td>Consumo frequente de alimentos calóricos.</td></tr>
    <tr><td><b>SMOKE</b></td><td>A pessoa é fumante?</td></tr>
    <tr><td><b>SCC</b></td><td>Realiza controle calórico?</td></tr>
  </tbody>
</table>

---

### 📊 Colunas Ordinais (Frequência)

<table>
  <thead>
    <tr>
      <th style="width:150px">Coluna</th>
      <th style="width:500px">Descrição</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>CAEC</b></td>
      <td>Consumo entre refeições: <code>Always</code>, <code>Frequently</code>, <code>Sometimes</code>, <code>no</code>.</td>
    </tr>
    <tr>
      <td><b>CALC</b></td>
      <td>Frequência de consumo de álcool: <code>Always</code>, <code>Frequently</code>, <code>Sometimes</code>, <code>no</code>.</td>
    </tr>
  </tbody>
</table>

---

### 🧬 Colunas Categóricas

<table>
  <thead>
    <tr>
      <th style="width:150px">Coluna</th>
      <th style="width:500px">Descrição</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><b>GENDER</b></td><td>Gênero: <code>Male</code>, <code>Female</code>.</td></tr>
    <tr><td><b>MTRANS</b></td><td>Meio de transporte: <code>Automobile</code>, <code>Bike</code>, <code>Motorbike</code>, <code>Public_Transportation</code>, <code>Walking</code>.</td></tr>
    <tr><td><b>OBESITY</b></td><td>Classificação do peso: <code>Insufficient_Weight</code>, <code>Normal_Weight</code>, <code>Overweight_Level_I</code>, <code>Overweight_Level_II</code>, <code>Obesity_Type_I</code>, <code>Obesity_Type_II</code>, <code>Obesity_Type_III</code>.</td></tr>
  </tbody>
</table>


In [ ]:
obesity_df.shape

In [ ]:
obesity_df.info()

In [ ]:
obesity_df.describe()

In [ ]:
obesity_df.isna().sum()

In [ ]:
obesity_df_norm = obesity_df.copy()


# Novas features
obesity_df_norm['IMC'] = obesity_df_norm['Weight'] / (obesity_df_norm['Height']**2)
obesity_df_norm['HealthyMealRatio'] = obesity_df_norm['FCVC'] / obesity_df_norm['NCP']
obesity_df_norm['ActivityBalance'] = obesity_df_norm['FAF'] - obesity_df_norm['TUE']
obesity_df_norm['TransportType'] = obesity_df_norm['MTRANS'].apply(lambda x: 'sedentary' if x in ['Automobile', 'Motorbike'] else 'active' if x in ['Bike', 'Walking'] else 'neutral')
obesity_df_norm['AgeGroup'] = (obesity_df_norm['Age']//10).astype(int)



# Transformando colunas categóricas em numéricas
bol_col = ['family_history', 'FAVC', 'SMOKE', 'SCC']
obesity_df_norm[bol_col] = obesity_df_norm[bol_col].replace({'yes': 1, 'no': 0})
obesity_df_norm[['Gender']] = obesity_df_norm[['Gender']].replace({'Male': 1, 'Female': 0})
obesity_df_norm = pd.get_dummies(obesity_df_norm, columns=['TransportType', 'CALC', 'CAEC']).replace(True, value=1).replace(False, value=0)


# Mapeando os tipos de obesidade
obesity_dict = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Obesity_Type_I': 2,
    'Obesity_Type_II': 3,
    'Obesity_Type_III': 4,
    'Overweight_Level_I': 5,
    'Overweight_Level_II': 6}

obesity_df_norm['Obesity'] = obesity_df_norm['Obesity'].map(obesity_dict)

# Calculando a pontuação de estilo de vida
obesity_df_norm['LifestyleScore'] = (
    (1 - obesity_df_norm['SMOKE']) +
    obesity_df_norm['SCC'] +
    (1 - obesity_df_norm['FAVC']) +
    (1 - obesity_df_norm['family_history'])
)


# Drop não numéricas
obesity_df_norm = obesity_df_norm.select_dtypes(exclude=['object'])

# Normalizando as colunas numéricas
mim_max_enc = MinMaxScaler()
min_max_col = ['Age', 'Height', 'Weight', 'IMC', 'ActivityBalance', 'HealthyMealRatio', 'AgeGroup']
obesity_df_norm[min_max_col] = mim_max_enc.fit_transform(obesity_df_norm[min_max_col])

obesity_df_norm


In [ ]:
obesity_df_norm_corr = obesity_df_norm.corr()
f, ax = plt.subplots(figsize=(25, 15))
sns.heatmap(obesity_df_norm_corr, annot=True, linewidths=.5, ax=ax)

In [3]:
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier


In [ ]:

best_global_model = None
best_accuracy = 0
best_model_name = ''
best_params = {}

# 1. Separar X e y
X = obesity_df_norm.drop(columns=['Obesity'])
y = obesity_df_norm['Obesity']

# 2. Divisão de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Modelos e hiperparâmetros
models = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'solver': ['lbfgs']
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [None, 10, 20]
        }
    },
    'SVM': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf']
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'max_depth': [None, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    }
}

# 4. Benchmark
results = []

for name, m in models.items():
    print(f"Treinando {name}...")
    grid = GridSearchCV(m['model'], m['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    # Salvar o melhor modelo global
    if acc > best_accuracy:
        best_accuracy = acc
        best_global_model = best_model
        best_model_name = name
        best_params = grid.best_params_

    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'Accuracy': acc,
        'MAE': mae,
        'F1 Macro': report['macro avg']['f1-score']
    })

# 5. Mostrar resultados
results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print(results_df)


In [1]:
from shared.utils import FeatureEngineering, TrasformNumeric, MinMaxScalerFeatures, LifestyleScore, ObesityMap, Model, DropNonNumeric
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import  MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin    
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.model_selection import RandomizedSearchCV

# Troque GridSearchCV por:


obesity_df = pd.read_csv('Obesity.csv', sep=',')


pipeline = Pipeline([
    ('feature_engineering', FeatureEngineering()),
    ('transform_numeric', TrasformNumeric()),
    ('min_max_scaler', MinMaxScalerFeatures()),
    ('dropnon_numeric', DropNonNumeric()),
    ('lifestyle_score', LifestyleScore()),
    ('model', Model())
])

X = obesity_df.drop(columns=['Obesity'])
y = ObesityMap().fit_transform(obesity_df['Obesity'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
f1 = classification_report(y_test, y_pred, output_dict=True)['macro avg']['f1-score']

print("\nDesempenho do modelo final (RandomForest):")
print(f"Acurácia: {acc:.4f}")
print(f"MAE: {mae:.4f}")
print(f"F1 Macro: {f1:.4f}")

Error in sys.excepthook:
Traceback (most recent call last):
  File "C:\Users\dhlui\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py", line 1934, in showtraceback
    stb = value._render_traceback_()
AttributeError: 'RuntimeError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\dhlui\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py", line 1936, in showtraceback
    stb = self.InteractiveTB.structured_traceback(etype,
  File "C:\Users\dhlui\AppData\Roaming\Python\Python39\site-packages\IPython\core\ultratb.py", line 1105, in structured_traceback
    return FormattedTB.structured_traceback(
  File "C:\Users\dhlui\AppData\Roaming\Python\Python39\site-packages\IPython\core\ultratb.py", line 999, in structured_traceback
    return VerboseTB.structured_traceback(
  File "C:\Users\dhlui\AppData\Roaming\Python\Python39


Desempenho do modelo final (RandomForest):
Acurácia: 0.9693
MAE: 0.0307
F1 Macro: 0.9687


c:\Users\dhlui\OneDrive\Área de Trabalho\Luis Henrique\ESTUDOS\FIAP\Tech Challenge\Fase 4\obesity-predict-model\shared\utils.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[bol_col] = X[bol_col].replace({'yes': 1, 'no': 0})
c:\Users\dhlui\OneDrive\Área de Trabalho\Luis Henrique\ESTUDOS\FIAP\Tech Challenge\Fase 4\obesity-predict-model\shared\utils.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X['Gender'] = X['Gender'].replace({'Male': 1, 'Female': 0})
c:\Users\dhlui\anaconda3\lib\site-packages\sklearn\pipeline.py:62: 